# 07_class_balance.ipynb

## Sprint 2 - PB-09: Balanceo de clases

En este notebook se analiza el desbalance de la variable objetivo y se aplican técnicas de balanceo únicamente sobre el conjunto de entrenamiento.

### Objetivo
Dejar el dataset preparado para modelado, evitando data leakage y manteniendo el test set con su distribución original.

### Input
- `data/interim/featured_data.csv`

### Output
- `data/processed/train_balanced.csv`
- `data/processed/test_original.csv`
- `data/processed/balance_summary.csv`

> Nota: la estandarización y el pipeline reproducible final se integrarán en la etapa de `08_pipeline.ipynb`.

## 0. Importación de librerías

In [3]:
import sys
!{sys.executable} -m pip install imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn]


In [4]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

## 1. Carga del dataset enriquecido

Se carga el dataset generado en la etapa de feature engineering (`PB-08`), que servirá como base para analizar el desbalance de clases.

In [5]:
df = pd.read_csv("../data/interim/featured_data.csv")
df_cb = df.copy()

print("Shape inicial:", df_cb.shape)
df_cb.head()

Shape inicial: (72983, 60)


,IsBadBuy,PurchDate,VehYear,VehicleAge,Make,Model,Trim,SubModel,WheelTypeID,VehOdo,...,WheelType_Special,odo_per_year,warranty_cost_ratio,cost_vs_acq_auction_avg_ratio,odometer_group,age_x_odo,cost_x_warranty,VehOdo_log1p,VehBCost_log1p,WarrantyCost_log1p
0,0,1970-01-01 00:00:01.260144000,2006.0,-36.0,0.161389,0.084158,0.112936,0.111111,1.0,89046.0,...,False,-2544.171429,0.156738,0.870525,alto,-3205656.0,7902300.0,11.396920,8.867991,7.015712
1,0,1970-01-01 00:00:01.260144000,2004.0,-34.0,0.103237,0.116258,0.115339,0.122195,1.0,93593.0,...,False,-2836.151515,0.138534,1.108680,alto,-3182162.0,8002800.0,11.446722,8.936035,6.960348
2,0,1970-01-01 00:00:01.260144000,2005.0,-35.0,0.103237,0.097606,0.098824,0.028336,2.0,73807.0,...,False,-2170.794118,0.283412,1.529816,medio_alto,-2583245.0,6806100.0,11.209222,8.497195,7.237059
3,0,1970-01-01 00:00:01.260144000,2004.0,-34.0,0.103237,0.234043,0.098824,0.122278,1.0,65617.0,...,False,-1988.393939,0.153621,2.164731,medio_bajo,-2230978.0,2583000.0,11.091605,8.318986,6.447306
4,0,1970-01-01 00:00:01.260144000,2005.0,-35.0,0.154091,0.171617,0.154472,0.160377,2.0,69367.0,...,False,-2040.205882,0.254936,1.021972,medio_bajo,-2427845.0,4080000.0,11.147181,8.294300,6.928538


## 2. Análisis de distribución del target

Se revisa la distribución de la variable objetivo para determinar si el dataset presenta desbalance de clases y si conviene aplicar alguna técnica de resampling.

In [6]:
target = "IsBadBuy"

print("Distribución absoluta:")
print(df_cb[target].value_counts())

print("\nDistribución relativa:")
print(df_cb[target].value_counts(normalize=True))

Distribución absoluta:
IsBadBuy
0    64007
1     8976
Name: count, dtype: int64

Distribución relativa:
IsBadBuy
0    0.877012
1    0.122988
Name: proportion, dtype: float64


## 3. Train/Test split

El split se realiza antes de cualquier técnica de balanceo para evitar data leakage.

Se utiliza `stratify=y` para conservar la proporción original de clases tanto en train como en test.

In [7]:
X = df_cb.drop(columns=[target])
y = df_cb[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train original:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nTest original:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Train original:
IsBadBuy
0    51205
1     7181
Name: count, dtype: int64
IsBadBuy
0    0.877008
1    0.122992
Name: proportion, dtype: float64

Test original:
IsBadBuy
0    12802
1     1795
Name: count, dtype: int64
IsBadBuy
0    0.87703
1    0.12297
Name: proportion, dtype: float64


## 4. Revisión de tipos de variables

Antes de seleccionar la técnica de balanceo, se revisa si el conjunto de entrenamiento contiene variables categóricas.

Esto es importante porque algunas técnicas, como SMOTE, funcionan mejor cuando las variables de entrada son completamente numéricas.

In [8]:
print(X_train.dtypes.value_counts())
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print("Columnas categóricas:", len(cat_cols))
print("Columnas numéricas:", len(num_cols))

float64    30
bool       21
object      7
int64       1
Name: count, dtype: int64
Columnas categóricas: 7
Columnas numéricas: 31


## 5. Comparación de técnicas de balanceo

Se comparan técnicas de resampling sobre el conjunto de entrenamiento:

- `RandomOverSampler`
- `RandomUnderSampler`
- `SMOTE` (solo si el conjunto es completamente numérico)

El objetivo es elegir una técnica razonable para esta etapa sin alterar el conjunto de prueba.

In [9]:
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

print("RandomOverSampler:")
print(y_train_ros.value_counts())

print("\nRandomUnderSampler:")
print(y_train_rus.value_counts())

RandomOverSampler:
IsBadBuy
1    51205
0    51205
Name: count, dtype: int64

RandomUnderSampler:
IsBadBuy
0    7181
1    7181
Name: count, dtype: int64


In [10]:
smote_result = None

if len(cat_cols) == 0:
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    print("\nSMOTE:")
    print(y_train_smote.value_counts())

    smote_result = (X_train_smote, y_train_smote)
else:
    print("SMOTE no se aplicó porque todavía existen variables categóricas.")

SMOTE no se aplicó porque todavía existen variables categóricas.


## 6. Selección de técnica final

Se selecciona la técnica que mejor se adapta al estado actual del dataset.

Criterios considerados:
- compatibilidad con los tipos de variables disponibles
- capacidad de corregir el desbalance
- simplicidad y trazabilidad para esta etapa del Sprint 2

In [11]:
if smote_result is not None:
    X_train_bal, y_train_bal = smote_result
    selected_method = "SMOTE"
else:
    X_train_bal, y_train_bal = X_train_ros, y_train_ros
    selected_method = "RandomOverSampler"

print("Técnica seleccionada:", selected_method)
print(y_train_bal.value_counts())
print(y_train_bal.value_counts(normalize=True))

Técnica seleccionada: RandomOverSampler
IsBadBuy
1    51205
0    51205
Name: count, dtype: int64
IsBadBuy
1    0.5
0    0.5
Name: proportion, dtype: float64


## 7. Comparación antes y después del balanceo

Se resume la distribución del target antes y después del resampling, manteniendo el conjunto de prueba sin modificar para una evaluación realista en la siguiente etapa.

In [12]:
summary = pd.DataFrame({
    "split": ["train_original", "train_balanced", "test_original"],
    "n_rows": [len(y_train), len(y_train_bal), len(y_test)],
    "minority_ratio": [
        y_train.mean(),
        y_train_bal.mean(),
        y_test.mean()
    ],
    "method": ["none", selected_method, "none"]
})

summary

,split,n_rows,minority_ratio,method
0,train_original,58386,0.122992,none
1,train_balanced,102410,0.500000,RandomOverSampler
2,test_original,14597,0.122970,none


## 8. Guardado del dataset procesado

Se guarda:
- el conjunto de entrenamiento balanceado
- el conjunto de prueba original
- un resumen del balance aplicado

Estos archivos serán utilizados en la siguiente etapa del Sprint 2 y quedarán versionados con DVC.

In [13]:
train_balanced = X_train_bal.copy()
train_balanced[target] = y_train_bal.values

test_original = X_test.copy()
test_original[target] = y_test.values

train_balanced.to_csv("../data/processed/train_balanced.csv", index=False)
test_original.to_csv("../data/processed/test_original.csv", index=False)
summary.to_csv("../data/processed/balance_summary.csv", index=False)

print("Archivos guardados en ../data/processed/")

Archivos guardados en ../data/processed/


## 9. Versionado con DVC

Los archivos procesados generados en esta etapa deben versionarse con DVC para asegurar trazabilidad y permitir que el Pipeline Builder trabaje sobre la versión correcta del dataset.